# Patch Geometrically Important Attention Heads

Selects heads that geometrically encode the operands/output (from the activation-analysis notebook's output) and patches all of them jointly between a base and source prompt. Unlike the single-head/layer sweeps, this patches every selected head at once, and by default across every sequence position rather than just the last token.

In [ ]:
%load_ext autoreload
%autoreload 2

## Setup

Locates the project root and imports `_config`/`_dataset`/`_mapping`/`_prompt` plus the intervention primitives used for the joint multi-head patching below.

In [ ]:
import gc
import os
import random
import sys
from collections import defaultdict

import pandas as pd
import torch
from tqdm import tqdm


def find_project_root(start_dir=None):
    path = os.path.abspath(start_dir or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(path, "src")) and os.path.isdir(os.path.join(path, "data")):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            raise RuntimeError("Could not find project root containing src/ and data/.")
        path = parent


project_root = find_project_root()
if project_root not in sys.path:
    sys.path.append(project_root)
if os.path.join(project_root, "src") not in sys.path:
    sys.path.append(os.path.join(project_root, "src"))

from src import _config, _dataset, _mapping, _prompt
from src._intervention import batch_intervene, forward_with_cache, get_attention_freeze_hooks

## Experiment Config

In [ ]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_stepwise",
    prompt_type="h_pre_penultimate_sum",
    data_root=f"{project_root}/data",
)
intervention_config = _config.InterventionConfig(
    intervention_loc="",  # restatement or reasoning or restatement_and_reasoning
    intervention_ids=[20],
    tok_pos_fn=_mapping.intervene_id_to_tok_pos_stepwise_3_digit_h,
)
attention_config = _config.AttentionFreezeConfig(
    enabled=False,
    num_attention=20,
    dataset_fn=_dataset.create_h_dataset,
    num_digits=3,
    prompt_fn=_prompt.get_stepwise_prompt,
    divide_num=22,
)

prompt_type = prompt_config.prompt_type
prompt_fn = attention_config.prompt_fn
freeze_attention = attention_config.enabled
num_attention = attention_config.num_attention
modifier_fn = lambda prompt, add_ds_entry: prompt

# Supported suffixes: *_prob, *_cos_sim, *_proj.
# *_prob selects top_k_heads; *_cos_sim and *_proj select heads above the thresholds below.
selection_metric = "operand_1_1_prob"
top_k_heads = 64
selection_thresholds = {
    "cos_sim": 0.45,
    "proj": 0.03,
}
head_activation_run_config = _config.RunConfig(
    experiment_root="experiments/attention_heads",
    result_dir="head_activation",
)
intervened_prob_csv = _config.build_run_output_path(
    prompt_config,
    _config.RunConfig(
        experiment_root=head_activation_run_config.experiment_root,
        result_dir=head_activation_run_config.result_dir,
        output_filename=f"intervened_logit_lens_{prompt_config.stem}_-1.csv",
    ),
)
intervened_similarity_csv = _config.build_run_output_path(
    prompt_config,
    _config.RunConfig(
        experiment_root=head_activation_run_config.experiment_root,
        result_dir=head_activation_run_config.result_dir,
        output_filename=f"intervened_residual_similarity_{prompt_config.stem}_-1.csv",
    ),
)

# Unlike head.ipynb/layer.ipynb (which patch only the last-token prediction site),
# heads.ipynb patches selected heads at every sequence position by default.
# Set to tok_pos_list (defined below) to instead restrict patching to the
# intervention token positions.
patch_tok_positions = None

run_config = _config.RunConfig(
    experiment_root="experiments/attention_heads",
    result_dir="geometric_heads",
)
batch_size = 24

## Load Model And Prompts

Loads the model/tokenizer, builds the (here disabled) attention-freeze hooks, and loads the configured base/source prompts.

In [ ]:
model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
def intervene_on_final_sum(prompt, add_ds_entry):
    source_prompt = prompt_fn(
        add_ds_entry["source_1_digits"],
        add_ds_entry["source_2_digits"],
        add_ds_entry["source_1_num"],
        add_ds_entry["source_2_num"],
    )
    return _prompt.get_intervened_prompt(intervention_ids, prompt, source_prompt)


modifier_fn = intervene_on_final_sum

In [ ]:
attention_freeze_hooks = _config.build_attention_freeze_hooks(
    model,
    tokenizer,
    attention_config,
    modifier_fn=modifier_fn,
)

In [ ]:
prompt_type_suffix = prompt_config.suffix
intervention_loc_suffix = intervention_config.location_suffix
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [ ]:
intervention_ids = _config.resolve_intervention_ids(
    prompt_config.model_type,
    prompt_config.prompt_type,
    intervention_config.intervention_loc,
    intervention_config.intervention_ids,
)
tok_pos_list = _config.build_tok_pos_list(intervention_config.tok_pos_fn, intervention_ids)
print(intervention_ids)
print(tok_pos_list)

[20]
[235]


## Select Geometrically Important Heads

Ranks heads by the chosen selection metric and keeps either the top-scoring heads or all heads above a threshold.

In [ ]:
def selection_suffix(metric):
    for suffix in ("cos_sim", "proj", "prob"):
        if metric.endswith(f"_{suffix}"):
            return suffix
    raise ValueError(f"Unsupported selection metric suffix for {metric!r}; expected *_prob, *_cos_sim, or *_proj")


metric_suffix = selection_suffix(selection_metric)
if metric_suffix == "prob":
    selection_csv = intervened_prob_csv
    selection_rule = f"top_{top_k_heads}"
    selection_threshold = None
else:
    selection_csv = intervened_similarity_csv
    selection_threshold = selection_thresholds[metric_suffix]
    selection_rule = f">{selection_threshold}"

selection_df = pd.read_csv(selection_csv)
if selection_metric not in selection_df.columns:
    raise ValueError(f"{selection_metric!r} not found in {selection_csv}")

ranked_heads_df = (
    selection_df[["layer", "head_num", selection_metric]]
    .sort_values(selection_metric, ascending=False)
    .drop_duplicates(subset=["layer", "head_num"])
    .reset_index(drop=True)
)
if metric_suffix == "prob":
    selected_heads_df = ranked_heads_df.head(top_k_heads)
else:
    selected_heads_df = ranked_heads_df[ranked_heads_df[selection_metric] > selection_threshold]

selected_heads = [
    (int(row["layer"]), int(row["head_num"]), float(row[selection_metric]))
    for _, row in selected_heads_df.iterrows()
]
heads_by_layer = defaultdict(list)
for layer, head, _ in selected_heads:
    heads_by_layer[layer].append(head)
heads_by_layer = {layer: sorted(heads) for layer, heads in sorted(heads_by_layer.items())}

if len(selected_heads) == 0:
    raise ValueError(f"No heads selected for {selection_metric} with rule {selection_rule}")

output_rule = f"top_{top_k_heads}" if metric_suffix == "prob" else f"gt_{selection_threshold:g}"
output_filename = f"patch_{output_rule}_{selection_metric}_{prompt_type}.csv"

print(f"selected {len(selected_heads)} heads from {selection_csv}")
print(f"selection_metric={selection_metric}, selection_rule={selection_rule}")
for layer, head, score in selected_heads:
    print(f"  layer {layer}, head {head}: {selection_metric}={score:.6f}")

selected 64 heads from /oscar/home/dkang33/arithmetic-reasoning-causality/experiments/attention_heads/output/GPT-OSS_stepwise/head_activation/intervened_logit_lens_h_pre_penultimate_sum_-1.csv
  layer 21, head 35: operand_1_1_prob=1.000000
  layer 23, head 23: operand_1_1_prob=1.000000
  layer 23, head 18: operand_1_1_prob=1.000000
  layer 23, head 61: operand_1_1_prob=1.000000
  layer 23, head 35: operand_1_1_prob=1.000000
  layer 23, head 34: operand_1_1_prob=1.000000
  layer 23, head 20: operand_1_1_prob=1.000000
  layer 23, head 16: operand_1_1_prob=1.000000
  layer 22, head 56: operand_1_1_prob=1.000000
  layer 23, head 43: operand_1_1_prob=1.000000
  layer 21, head 31: operand_1_1_prob=1.000000
  layer 23, head 42: operand_1_1_prob=1.000000
  layer 23, head 40: operand_1_1_prob=1.000000
  layer 22, head 60: operand_1_1_prob=1.000000
  layer 23, head 17: operand_1_1_prob=1.000000
  layer 23, head 7: operand_1_1_prob=1.000000
  layer 23, head 19: operand_1_1_prob=1.000000
  layer 2

## Multi-Head Patching Helpers

Defines the hooks that copy the source prompt's activations into the base prompt for exactly the selected heads.

In [ ]:
def _aligned_tokenize(base_prompts, source_prompts):
    """Tokenize base/source prompts together so cached source activations align with base inputs."""
    batch_size = len(base_prompts)
    tokens = tokenizer(
        base_prompts + source_prompts,
        add_special_tokens=False,
        return_tensors="pt",
        padding=True,
        padding_side="left",
    ).to(model.device)
    base_tokens = {key: value[:batch_size] for key, value in tokens.items()}
    source_tokens = {key: value[batch_size:] for key, value in tokens.items()}
    return base_tokens, source_tokens


def get_batch_multihead_intervene_hook(source_activation, head_idxs, num_heads, tok_positions=None):
    head_idxs = list(head_idxs)

    def intervene_hook(module, inputs):
        hidden_states = inputs[0]
        original_shape = hidden_states.shape
        if hidden_states.shape != source_activation.shape:
            raise ValueError(
                f"Base/source activation shapes differ: {hidden_states.shape} vs {source_activation.shape}"
            )
        if original_shape[-1] % num_heads != 0:
            raise ValueError(f"o_proj input dim {original_shape[-1]} is not divisible by {num_heads} heads")

        head_dim = original_shape[-1] // num_heads
        positions = list(range(original_shape[1])) if tok_positions is None else tok_positions

        patched = hidden_states.view(original_shape[0], original_shape[1], num_heads, head_dim)
        source = source_activation.view(original_shape[0], original_shape[1], num_heads, head_dim)
        for head_idx in head_idxs:
            patched[:, positions, head_idx, :] = source[:, positions, head_idx, :]
        return (patched.view(original_shape),)

    return intervene_hook


def prepare_batch_multihead_intervention(model, base_prompts, source_prompts, heads_by_layer, tok_positions=None):
    base_tokens, source_tokens = _aligned_tokenize(base_prompts, source_prompts)
    module_names = [f"model.layers.{layer}.self_attn.o_proj" for layer in heads_by_layer]

    _, cache = forward_with_cache(
        model,
        source_tokens["input_ids"],
        module_names=module_names,
        attention_mask=source_tokens["attention_mask"],
        pre_hook=True,
    )

    num_heads = model.config.num_attention_heads
    hooks = []
    for layer, head_idxs in heads_by_layer.items():
        module_name = f"model.layers.{layer}.self_attn.o_proj"
        hooks.append({
            module_name: get_batch_multihead_intervene_hook(
                cache[module_name],
                head_idxs,
                num_heads,
                tok_positions=tok_positions,
            )
        })

    del cache
    torch.cuda.empty_cache()
    gc.collect()
    return base_tokens, source_tokens, hooks


def first_token_ids(strings):
    return tokenizer(
        [str(value) for value in strings],
        add_special_tokens=False,
        return_tensors="pt",
    )["input_ids"][:, 0]


def label_strings(batch_rows, output_col, fallback_col):
    if output_col in batch_rows.columns and pd.notna(batch_rows.iloc[0][output_col]) and batch_rows.iloc[0][output_col] != "":
        return batch_rows[output_col].tolist()
    return batch_rows[fallback_col].tolist()

## Patch All Selected Heads At Once

Runs the joint intervention on each prompt batch and records the generated answer and factual/counterfactual probabilities.

In [ ]:
if freeze_attention:
    raise NotImplementedError("Joint geometric-head patching currently supports freeze_attention=False.")

patched_heads_str = "; ".join(f"{layer}:{','.join(str(head) for head in heads)}" for layer, heads in heads_by_layer.items())
header = list(prompts.columns) + [
    "intervention_ids",
    "intervention_id",
    "selection_csv",
    "selection_metric",
    "selection_rule",
    "selection_threshold",
    "top_k_heads",
    "patched_heads",
    "num_patched_heads",
    "generated_text",
    "factual_label_probability",
    "counterfactual_label_probability",
]
patch_run_config = _config.RunConfig(
    experiment_root=run_config.experiment_root,
    result_dir=run_config.result_dir,
    output_filename=output_filename,
)
filepath = _config.build_run_output_filepath(prompt_config, patch_run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    batch_rows = prompts.iloc[i:i + batch_size]
    cur_batch_size = len(batch_rows)

    factual_labels = first_token_ids(label_strings(batch_rows, "factual_output", "base_sum"))
    counterfactual_labels = first_token_ids(label_strings(batch_rows, "counterfactual_output", "source_sum"))

    base_prompts = batch_rows["base_prompt"].tolist()
    source_prompts = batch_rows["source_prompt"].tolist()
    tokens, source_tokens, intervene_hooks = prepare_batch_multihead_intervention(
        model,
        base_prompts,
        source_prompts,
        heads_by_layer,
        tok_positions=patch_tok_positions,
    )
    input_length = tokens["input_ids"].shape[1]

    with torch.no_grad():
        output = batch_intervene(
            model,
            tokens["input_ids"],
            intervene_hooks + attention_freeze_hooks,
            attention_mask=tokens["attention_mask"],
        )

    logits = output.logits[:, -1, :]
    pred_toks = logits.argmax(dim=-1)
    probs = torch.nn.functional.softmax(logits, dim=-1)
    factual_labels = factual_labels.to(probs.device)
    counterfactual_labels = counterfactual_labels.to(probs.device)
    batch_idx = torch.arange(cur_batch_size, device=probs.device)
    factual_prob = probs[batch_idx, factual_labels]
    counterfactual_prob = probs[batch_idx, counterfactual_labels]
    tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)

    for j, (_, row) in enumerate(batch_rows.iterrows()):
        generated_text = tokenizer.decode(tokens["input_ids"][j, input_length:], skip_special_tokens=True)
        _config.write_to_csv(filepath, row.to_list() + [
            intervention_ids,
            ", ".join(str(id) for id in intervention_ids),
            selection_csv,
            selection_metric,
            selection_rule,
            selection_threshold,
            top_k_heads if metric_suffix == "prob" else "",
            patched_heads_str,
            len(selected_heads),
            generated_text,
            factual_prob[j].item(),
            counterfactual_prob[j].item(),
        ])

    del tokens, source_tokens, intervene_hooks, output, logits, probs
    torch.cuda.empty_cache()
    gc.collect()

print(f"wrote {filepath}")

  0%|                                                                               | 0/11 [00:00<?, ?it/s]

100%|██████████████████████████████████████████████████████████████████████| 11/11 [01:40<00:00,  9.13s/it]

wrote /oscar/home/dkang33/arithmetic-reasoning-causality/experiments/attention_heads/output/GPT-OSS_stepwise/geometric_heads/patch_top_64_operand_1_1_prob_h_pre_penultimate_sum_1.csv
